In [3]:
from ollama import Client

# === Connexion au modèle local via Ollama ===
client = Client(host='http://localhost:11434')

# === Schéma de base ERP ===
erp_schema = {
    "tables": {
        "commandes": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "date_commande": "DATE",
                "montant": "DECIMAL(10,2)",
                "client_id": "INT FOREIGN KEY",
                "produit_id": "INT FOREIGN KEY"
            },
            "description": "Commandes passées par les clients"
        },
        "clients": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "secteur": "VARCHAR(50)",
                "ville": "VARCHAR(50)"
            },
            "description": "Informations clients"
        },
        "produits": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "categorie": "VARCHAR(50)",
                "prix_unitaire": "DECIMAL(10,2)"
            },
            "description": "Catalogue produits"
        },
        "fournisseurs": {
            "columns": {
                "id": "INT PRIMARY KEY",
                "nom": "VARCHAR(100)",
                "pays": "VARCHAR(50)",
                "categorie_fournisseur": "VARCHAR(50)"
            },
            "description": "Fournisseurs disponibles"
        }
    },
    "relations": {
        "commandes.client_id": "clients.id",
        "commandes.produit_id": "produits.id"
    }
}

# === Formateur de schéma lisible ===
def format_schema_detailed(schema):
    tables_desc = []
    for table, info in schema["tables"].items():
        cols = ", ".join([f"{col} ({type_col})" for col, type_col in info["columns"].items()])
        tables_desc.append(f"Table {table}: {cols} - {info['description']}")
    relations = "\nRelations:\n" + "\n".join([f"- {rel}" for rel in schema["relations"]])
    return "\n".join(tables_desc) + relations

# === Générateur de prompt SQL ===
def generate_sql_prompt(analysis_text: str, schema: dict) -> str:
    return f"""
Tu es un assistant SQL expert. Ton rôle est de transformer une analyse NLP structurée en requête SQL **exécutable et propre**, à partir du schéma ERP suivant :

📊 SCHÉMA :
{format_schema_detailed(schema)}

🧠 ANALYSE STRUCTURÉE :
{analysis_text}

🎯 OBJECTIF :
Génère la requête SQL correspondante, en respectant ces règles :

- Utilise SEULEMENT les colonnes mentionnées dans "COLONNES"
- Applique TOUS les filtres listés dans "FILTRES"
- Implémente toutes les jointures avec INNER JOIN
- Pour les dates, utilise EXTRACT(MONTH...) et EXTRACT(YEAR...) si mentionné
- Ne rajoute AUCUNE colonne ou table en plus
- Si AGRÉGATION est mentionnée, utilise COUNT/GROUP BY selon le cas
- Écris une requête SQL propre, indentée, sans explication
- N’ajoute pas d’agrégation s’il n’y en a pas dans la section AGRÉGATION 
- "Place TOUJOURS les conditions EXTRACT(...) dans une clause WHERE, jamais dans SELECT."

📦 Format de sortie :
```sql
-- requête SQL ici
```
Commence maintenant.
""".strip()

# === Appel Ollama pour SQL ===
def generer_sql_via_ollama(analysis_text):
    prompt = generate_sql_prompt(analysis_text, erp_schema)
    response = client.chat(model='llama3.2', messages=[
        {"role": "user", "content": prompt}
    ])
    return response['message']['content']

# === Exemple d'utilisation ===
analyse_nlp = """
INTENTION: SELECT
TABLES: [commandes, clients]
COLONNES: [commandes.id, commandes.date_commande]
FILTRES: [EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3, EXTRACT(YEAR FROM commandes.date_commande) = 2024]
JOINTURES: [commandes.client_id = clients.id]
AGRÉGATION:
"""

# Génération SQL
sql_output = generer_sql_via_ollama(analyse_nlp)
print("\n💾 Requête SQL générée :\n")
print(sql_output)



💾 Requête SQL générée :

-- Requête SQL générée en fonction des critères donnés :

SELECT 
  commandes.id,
  commandes.date_commande,
  clients.nom AS nom_client,
  produits.nom AS nom_produit,
  COUNT(*) AS nb_commandes
FROM 
  commandes
  INNER JOIN clients ON commandes.client_id = clients.id
  INNER JOIN produits ON commandes.produit_id = produits.id
WHERE 
  EXTRACT(MONTH FROM commandes.date_commande) BETWEEN 1 AND 3
  AND EXTRACT(YEAR FROM commandes.date_commande) = 2024
